# Notebook 01: Recommendation Fundamentals

## Why This Matters

Netflix saves an estimated $1 billion per year through recommendations — 80% of content watched on the platform is discovered via its recommendation engine, not search. Spotify's Discover Weekly generates more streams than any individual playlist and has been credited with keeping users on the platform longer than any other feature. YouTube's recommendation engine drives 70% of all time watched. Recommender systems are among the highest-ROI investments a product company can make, and they sit at the intersection of machine learning, statistics, and product thinking. This notebook lays the conceptual and practical foundation for everything that follows in this series.

## Background

A recommender system answers one question: **given what I know about this user, what should I show them next?** There are three broad paradigms:

1. **Collaborative Filtering (CF)** — Use the behavior of many users to recommend to one user. The core insight: if Alice and Bob both liked films A, B, and C, then if Bob also liked D, Alice probably will too.
   - *User-based CF* (memory-based): Find users similar to the target user, aggregate their ratings. Simple, interpretable, but doesn't scale — similarity must be recomputed as the user base grows. Amazon experimented with this in the 1990s and quickly hit walls.
   - *Item-based CF*: Precompute item-item similarities offline. Amazon's breakthrough (Linden et al., 2003): "Customers who bought X also bought Y." Item catalog grows slower than the user base, so the similarity matrix is more stable and cacheable.

2. **Content-Based Filtering** — Represent items by features (genre, director, tempo, tags) and recommend items similar to what the user has liked. Advantage: works for new items (no cold start on the item side). Disadvantage: only recommends things similar to what the user already likes — the "filter bubble" problem.

3. **Hybrid Systems** — Every major production system combines both. Netflix uses content features to bootstrap new items and collaborative signals to personalize at scale. The two signals are complementary: CF captures community taste, content captures item identity.

### The Interaction Matrix

The foundation of most recommender systems is the **user-item interaction matrix** $R \in \mathbb{R}^{m \times n}$ where $m$ is the number of users and $n$ is the number of items. Most entries are missing — a user has only seen a tiny fraction of the catalog. Typical sparsity: 99.9%+ missing.

**Explicit feedback**: Star ratings (1–5), thumbs up/down. Clean signal, hard to collect at scale. Users rate ~1% of items they consume.

**Implicit feedback**: Clicks, views, purchases, listen-time, search queries. Noisy (a click doesn't mean you liked it), but abundant. Most production systems run on implicit data.

### The Cold Start Problem

- **New user**: no history → can't personalize. Solutions: onboarding questionnaire, demographic-based priors, popular items.
- **New item**: no interactions → won't surface in CF. Solutions: content-based warm start, injection into exploration slots.
- **New platform**: no users, no items, no data. Solutions: import social graph, use public datasets.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn requests

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import requests, zipfile, io, os, warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Ready.")

## 1. Loading MovieLens Data

MovieLens is the benchmark dataset for recommender systems research, published by GroupLens at the University of Minnesota. The 100K variant has 100,000 ratings from 943 users on 1,682 movies — small enough for fast experimentation, well-studied enough to compare against published baselines.

We'll use the explicit 1–5 star ratings here. In later notebooks (03 onward) we'll treat these as implicit signals.

In [ ]:
# Download MovieLens 100K
ML_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = "/tmp/ml-100k"

if not os.path.exists(DATA_DIR):
    print("Downloading MovieLens 100K...")
    r = requests.get(ML_URL, timeout=60)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall("/tmp/")
    print("Done.")
else:
    print("Already downloaded.")

# Load ratings: user_id | item_id | rating | timestamp
ratings = pd.read_csv(
    f"{DATA_DIR}/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

# Load movie metadata
movies = pd.read_csv(
    f"{DATA_DIR}/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names=["item_id", "title", "release_date", "video_release_date",
           "imdb_url"] + [f"genre_{i}" for i in range(19)]
)
movies = movies[["item_id", "title"]]

print(f"Ratings shape: {ratings.shape}")
print(f"Users: {ratings.user_id.nunique()}, Items: {ratings.item_id.nunique()}")
print(f"Rating range: {ratings.rating.min()} – {ratings.rating.max()}")
print(f"Sparsity: {1 - len(ratings) / (ratings.user_id.nunique() * ratings.item_id.nunique()):.4f}")
ratings.head()

## 2. Building the User-Item Interaction Matrix

The interaction matrix $R$ has users as rows and items as columns. Entry $R_{ui}$ is the rating user $u$ gave item $i$, or 0 if they haven't rated it. We'll build both a dense NumPy array (for visualization) and a sparse CSR matrix (for efficient computation).

Key insight: the matrix is extremely sparse. Most algorithms only operate on *observed* entries during training — treating missing as "unknown" rather than zero is critical.

In [ ]:
# Build user-item matrix (dense for small dataset)
n_users = ratings.user_id.max()
n_items = ratings.item_id.max()

# Dense matrix — only feasible for small datasets
R_dense = np.zeros((n_users, n_items), dtype=np.float32)
for _, row in ratings.iterrows():
    R_dense[int(row.user_id) - 1, int(row.item_id) - 1] = row.rating

print(f"Matrix shape: {R_dense.shape}")
print(f"Non-zero entries: {np.count_nonzero(R_dense):,}")
print(f"Total entries: {R_dense.size:,}")
print(f"Sparsity: {1 - np.count_nonzero(R_dense) / R_dense.size:.4f}")

# Sparse matrix for efficient computation
R_sparse = csr_matrix(R_dense)

# Visualize a 50x50 submatrix
fig, ax = plt.subplots(figsize=(8, 6))
subset = R_dense[:50, :100]
im = ax.imshow(subset, aspect="auto", cmap="YlOrRd",
               norm=mcolors.BoundaryNorm([0,1,2,3,4,5], 256))
ax.set_xlabel("Item ID (first 100)")
ax.set_ylabel("User ID (first 50)")
ax.set_title("User-Item Rating Matrix (50 users × 100 items)\nWhite = unrated (missing), Yellow–Red = 1–5 stars")
plt.colorbar(im, ax=ax, label="Rating")
plt.tight_layout()
plt.savefig("/tmp/interaction_matrix.png", dpi=120)
plt.show()
print("White cells = unrated (99%+ of all entries)")

## 3. Item-Item Cosine Similarity

Item-based CF computes the similarity between every pair of items based on *how users have rated them*. Two items are similar if users who rated one tend to rate the other similarly. This is the core of Amazon's "customers also bought" feature.

**Why cosine over Pearson here?** For binary/sparse data, cosine similarity handles zero-heavy vectors gracefully. Pearson correlation is better when you want to account for user rating bias (some users always rate high).

$$\text{sim}(i, j) = \frac{\mathbf{r}_i \cdot \mathbf{r}_j}{\|\mathbf{r}_i\| \|\mathbf{r}_j\|}$$

where $\mathbf{r}_i$ is the column vector of all ratings given to item $i$.

In [ ]:
# Compute item-item cosine similarity
# R_dense columns = items, rows = users
# We need similarity between items => compare columns

print("Computing item-item cosine similarity on first 500 items...")
item_matrix = R_dense[:, :500].T  # shape: (500 items, 943 users)

# sklearn handles sparse/zero vectors gracefully
item_sim = cosine_similarity(item_matrix)  # shape: (500, 500)
np.fill_diagonal(item_sim, 0)  # zero out self-similarity

print(f"Item similarity matrix shape: {item_sim.shape}")
print(f"Similarity range: {item_sim.min():.4f} – {item_sim.max():.4f}")
print(f"Mean similarity (non-zero): {item_sim[item_sim > 0].mean():.4f}")

# Visualize similarity heatmap for first 50 items
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(item_sim[:50, :50], cmap="Blues", aspect="auto")
ax.set_title("Item-Item Cosine Similarity (first 50 items)")
ax.set_xlabel("Item ID")
ax.set_ylabel("Item ID")
plt.tight_layout()
plt.savefig("/tmp/item_similarity.png", dpi=120)
plt.show()

## 4. Generating Recommendations with Item-Item CF

Given a target user, item-based CF works like this:
1. Find all items the user has rated.
2. For each rated item, find the top-K most similar items.
3. Score unrated items by a weighted sum of the user's ratings, weighted by item similarity.
4. Return the top-N scored items.

$$\hat{r}_{ui} = \frac{\sum_{j \in \mathcal{N}(i)} \text{sim}(i,j) \cdot r_{uj}}{\sum_{j \in \mathcal{N}(i)} |\text{sim}(i,j)|}$$

This is interpretable and doesn't require learning parameters — but it doesn't generalize beyond the observed items and users.

In [ ]:
def item_cf_recommend(user_id, R, item_sim, movies_df, n_recs=10, n_similar=20):
    """Item-based CF recommendations for a user."""
    user_idx = user_id - 1
    user_ratings = R[user_idx]  # shape: (n_items,)
    rated_mask = user_ratings > 0
    rated_items = np.where(rated_mask)[0]

    if len(rated_items) == 0:
        return pd.DataFrame({"message": ["No ratings found — cold start!"]})

    # Score each unrated item: weighted sum of similarities to rated items
    scores = np.zeros(R.shape[1])
    sim_sums = np.zeros(R.shape[1])

    for item_idx in rated_items:
        if item_idx >= item_sim.shape[0]:
            continue
        sims = item_sim[item_idx].copy()
        sims[rated_mask[:item_sim.shape[0]]] = 0  # don't recommend already-rated
        top_n_mask = np.argsort(sims)[-n_similar:]
        scores[top_n_mask] += sims[top_n_mask] * user_ratings[item_idx]
        sim_sums[top_n_mask] += np.abs(sims[top_n_mask])

    # Normalize
    with np.errstate(divide="ignore", invalid="ignore"):
        pred = np.where(sim_sums > 0, scores / sim_sums, 0)

    # Mask already-rated and items outside our similarity matrix
    pred[rated_mask] = 0
    pred[item_sim.shape[0]:] = 0

    top_items = np.argsort(pred)[::-1][:n_recs]

    recs = pd.DataFrame({
        "item_id": top_items + 1,
        "predicted_rating": pred[top_items],
    }).merge(movies_df, on="item_id")

    return recs[["item_id", "title", "predicted_rating"]]

# Demonstrate for a few users
for uid in [1, 42, 100]:
    recs = item_cf_recommend(uid, R_dense, item_sim, movies, n_recs=5)
    print(f"\n=== Top-5 recommendations for User {uid} ===")
    # Show what they've already rated highly
    user_ratings = ratings[ratings.user_id == uid].merge(movies, on="item_id")
    top_rated = user_ratings.nlargest(3, "rating")[["title", "rating"]]
    print(f"User {uid} top-rated: {list(zip(top_rated.title.values, top_rated.rating.values))}")
    print(recs.to_string(index=False))

## 5. Explicit vs Implicit Feedback Distributions

Understanding the distribution of your feedback signal shapes every modeling decision. Explicit ratings follow a left-skew J-curve (users rate things they feel strongly about), while implicit signals are power-law distributed (a few items get most interactions).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Explicit: rating distribution
rating_counts = ratings["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Explicit Feedback: Rating Distribution\n(J-curve: users rate things they feel strongly about)")
for i, (r, c) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0].text(r, c + 500, f"{c:,}", ha="center", fontsize=9)

# Implicit proxy: number of ratings per item (popularity distribution)
item_pop = ratings.groupby("item_id").size().sort_values(ascending=False).reset_index()
item_pop.columns = ["item_id", "n_ratings"]
axes[1].plot(range(len(item_pop)), item_pop.n_ratings.values, color="coral")
axes[1].fill_between(range(len(item_pop)), item_pop.n_ratings.values, alpha=0.3, color="coral")
axes[1].set_xlabel("Item rank (by popularity)")
axes[1].set_ylabel("Number of ratings")
axes[1].set_title("Item Popularity (Power-Law Distribution)\nTop 10% items get 50%+ of all interactions")
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig("/tmp/feedback_distributions.png", dpi=120)
plt.show()

top10_items = item_pop.head(10).merge(movies, on="item_id")
print("\nMost-rated items (popularity bias challenge):")
print(top10_items[["title", "n_ratings"]].to_string(index=False))

## 6. Cold Start Visualization

Cold start is one of the most underestimated practical challenges. New users and new items enter the system constantly — Netflix adds thousands of new titles per year, Spotify has 100,000 new tracks uploaded per day. Without a strategy, new items get zero exposure and new users get generic recommendations.

In [ ]:
# Visualize the cold start distribution
ratings_per_user = ratings.groupby("user_id").size()
ratings_per_item = ratings.groupby("item_id").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# User activity distribution
axes[0].hist(ratings_per_user.values, bins=50, color="mediumseagreen", edgecolor="white", log=True)
axes[0].axvline(5, color="red", linestyle="--", label="Cold start threshold (5 ratings)")
axes[0].set_xlabel("Ratings per user")
axes[0].set_ylabel("Number of users (log scale)")
axes[0].set_title("User Activity Distribution\nMany users have very few ratings")
axes[0].legend()

cold_users = (ratings_per_user < 5).sum()
print(f"Users with < 5 ratings (cold start): {cold_users} ({cold_users/len(ratings_per_user):.1%})")

# Item interaction distribution
axes[1].hist(ratings_per_item.values, bins=50, color="mediumpurple", edgecolor="white", log=True)
axes[1].axvline(10, color="red", linestyle="--", label="Cold start threshold (10 ratings)")
axes[1].set_xlabel("Ratings per item")
axes[1].set_ylabel("Number of items (log scale)")
axes[1].set_title("Item Popularity Distribution\nLong tail of rarely-rated items")
axes[1].legend()

cold_items = (ratings_per_item < 10).sum()
print(f"Items with < 10 ratings (cold start): {cold_items} ({cold_items/len(ratings_per_item):.1%})")

plt.tight_layout()
plt.savefig("/tmp/cold_start.png", dpi=120)
plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Collaborative Filtering | Users with similar behavior like similar things — no item features needed |
| User-based CF | Find k nearest users, aggregate their ratings — doesn't scale to millions of users |
| Item-based CF | Precompute item similarity offline — Amazon's breakthrough (2003), more stable |
| Content-based filtering | Use item features — solves new item cold start, creates filter bubble |
| Hybrid systems | Every major production system (Netflix, Spotify, YouTube) combines multiple signals |
| Interaction matrix | Users × items, 99%+ sparse — most algorithms only train on observed entries |
| Explicit feedback | Star ratings — clean signal, rare (users rate ~1% of consumed items) |
| Implicit feedback | Clicks, views, purchases — noisy but abundant — powers most production systems |
| Cold start | New users and new items have no history — requires special handling |
| Popularity bias | Power-law distribution — top 1% items dominate interactions |

### Common Pitfalls
- Treating missing entries as zero ratings — they are *unknown*, not negative
- Ignoring the cold start problem until production — design for it from day one
- Over-relying on explicit ratings — implicit feedback is noisier but far more abundant
- Building item-based CF on the full catalog without approximate NN — $O(n^2)$ doesn't scale
- Evaluating only on popular items — tail item performance matters for discovery and fairness